# 4. 워크플로우 설계 & Tool 연동

**LangGraph**의 조건 분기, Tool 바인딩, 에러 핸들링을 활용한 **실전 에이전트 워크플로** 설계:
- **조건 분기(Conditional Edge)** — 상태에 따라 다른 노드로 라우팅하는 브랜칭 로직
- **Tool 정의 & 바인딩** — `@tool` 데코레이터와 `bind_tools()`로 LLM에 도구 연결
- **Tool 결합 워크플로우** — 검색, 계산, API 호출 등 복합 도구를 그래프에 통합
- **ReAct Agent** — Tool-calling 루프를 Graph API로 명시적 구현
- **에러 핸들링** — 체크포인터, Durable Execution으로 안전한 워크플로 구성
- **HITL (Human-in-the-Loop)** — `interrupt()`와 `Command(resume=...)`로 Agent 제어 흐름 설계
- **Agent 디버깅** — 스트리밍, 트레이싱으로 실행 흐름 실시간 추적 및 디버깅

### 학습 목표
- LangGraph의 조건 분기와 Tool 바인딩으로 ReAct Agent를 구현할 수 있다
- 체크포인터와 Durable Execution으로 안전한 워크플로를 설계할 수 있다
- **HITL 패턴과 체크포인터를 활용한 Agent 제어 흐름을 설계할 수 있다**
- 스트리밍과 트레이싱을 활용하여 Agent 실행을 디버깅할 수 있다

## 0) 환경 설정

In [1]:
# [4-1] : 라이브러리 설치
# [핵심] LangGraph + Tool 연동에 필요한 전체 패키지 — 첫 실행 시 한 번만 필요
# [주의] 버전 충돌 시 런타임 재시작 후 재실행

!uv pip install -q langchain langchain-core langchain-openai langchain-community langgraph langgraph-checkpoint python-dotenv

In [2]:
# [4-2] : 환경변수 로드 및 API 키 검증
# [핵심] .env 파일에서 API 키를 로드 — 하드코딩 금지 원칙
# [주의] override=True → 셀 재실행 시 .env 값으로 덮어씀

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."

print(f"OPENAI_API_KEY: {'OK' if OPENAI_API_KEY else 'MISSING'} (필수)")

OPENAI_API_KEY: OK (필수)


In [3]:
# [4-4] : 공통 LLM 모델 초기화
# [핵심] 노트북 전체에서 재사용할 모델을 한 곳에서 관리
# [주의] temperature=0 → 결정적 출력; 실습 재현성을 위해 고정

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1", temperature=0)
print(f"LLM 준비 완료: {llm.model_name}")

LLM 준비 완료: gpt-4.1


---

## 1) Tool 정의 & 바인딩 — @tool, bind_tools(), ToolMessage

LangChain의 `@tool` 데코레이터를 사용하면 일반 Python 함수를 LLM이 호출할 수 있는 **도구(Tool)**로 변환할 수 있습니다.

- `@tool` — 함수의 docstring이 도구 설명이 되어 LLM이 언제 사용할지 판단
- `bind_tools()` — 도구 스키마를 모델에 바인딩하여 LLM이 적절한 시점에 도구를 선택
- `ToolMessage` — 도구 실행 결과를 LLM에게 전달하는 메시지 타입

In [ ]:
# [4-7] : @tool 데코레이터로 도구 정의
# [핵심] docstring이 LLM에게 전달되는 도구 설명 — 명확하게 작성 필수
# [주의] 타입 힌트가 도구 스키마의 파라미터 타입이 됨

from langchain_core.tools import tool


@tool
def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return 


@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱합니다."""
    return 


@tool
def divide(a: float, b: float) -> float:
    """a를 b로 나눕니다. b가 0이면 에러를 반환합니다."""
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return 


tools = 

In [ ]:
# [4-8] : bind_tools()로 LLM에 도구 바인딩
# [핵심] 바인딩 후 LLM은 응답에 tool_calls를 포함시켜 도구 호출을 요청
# [주의] bind_tools()는 새 모델 인스턴스를 반환 — 원본은 변경하지 않음

from langchain_core.messages import HumanMessage, ToolMessage, AnyMessage

# 도구를 모델에 바인딩
llm_with_tools = 

# 도구가 필요한 질문
response = 


LLM 응답 타입: AIMessage
content: ''
tool_calls: [{'name': 'multiply', 'args': {'a': 7, 'b': 8}, 'id': 'call_6OI2kY3lBw2wCpAsoGZ2ce4u', 'type': 'tool_call'}]


In [6]:
# [4-9] : tool_calls 수동 실행 & ToolMessage 구성
# [핵심] LLM의 tool_calls를 실제로 실행하고 결과를 ToolMessage로 변환
# [주의] tool_call_id가 일치해야 LLM이 어떤 호출의 결과인지 매핑 가능

tools_by_name = {t.name: t for t in tools}  # 이름으로 빠르게 조회

# tool_calls를 순회하며 실행
tool_messages = []
for tc in response.tool_calls:
    tool_fn = tools_by_name[tc["name"]]         # 도구 함수 조회
    result = tool_fn.invoke(tc["args"])           # 도구 실행
    tool_msg = ToolMessage(
        content=str(result),
        tool_call_id=tc["id"],                   # 호출 ID 매핑 필수
    )
    tool_messages.append(tool_msg)
    print(f"도구 실행: {tc['name']}({tc['args']}) → {result}")

# 전체 메시지 체인 구성: Human → AI(tool_calls) → ToolMessage
full_messages = [
    HumanMessage(content="7 곱하기 8은 얼마인가요?"),
    response,           # AI 메시지 (tool_calls 포함)
    *tool_messages,     # 도구 결과
]

# LLM이 도구 결과를 보고 최종 답변 생성
final_response = llm_with_tools.invoke(full_messages)
print(f"\n최종 답변: {final_response.content}")

도구 실행: multiply({'a': 7, 'b': 8}) → 56.0

최종 답변: 7 곱하기 8은 56입니다.


---

## 2) Tool 결합 워크플로우 — search / calculator / API tools in graph

실전에서는 **여러 종류의 도구**를 하나의 그래프에 통합합니다:
- **검색(Search)** — 외부 지식 검색
- **계산(Calculator)** — 수학 연산
- **API 호출** — 외부 서비스 연동

LLM이 질문에 따라 적절한 도구를 선택하고, 결과를 조합하여 최종 답변을 생성합니다.



In [7]:
# [4-10] : 다양한 종류의 도구 정의 — 검색, 계산, API
# [핵심] 실전에서 자주 사용하는 도구 패턴 — 각 docstring이 LLM의 도구 선택 기준
# [주의] 실습용 모의(mock) 구현 — 프로덕션에서는 실제 API로 교체

import json


@tool
def search_knowledge(query: str) -> str:
    """SK 하이닉스 내부 지식 베이스를 검색합니다. 반도체, 메모리 제품 정보 등을 조회할 때 사용합니다."""
    # 모의 지식 베이스
    knowledge = {
        "반도체": "SK하이닉스는 세계적인 메모리 반도체 제조사로, HBM(고대역폭 메모리) 시장을 선도합니다.",
        "HBM": "HBM3E는 최신 고대역폭 메모리로, AI 가속기에 필수적인 부품입니다. 최대 36GB 용량을 지원합니다.",
        "AI": "AI 반도체 시장은 2025년 기준 연간 30% 이상 성장하고 있습니다.",
    }
    for key, value in knowledge.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 검색 결과가 없습니다."


@tool
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. 사칙연산, 거듭제곱(**) 등을 지원합니다."""
    try:
        # 안전한 수식 평가 — 빌트인 함수를 제한하여 임의 코드 실행 방지
        allowed = {"__builtins__": {}, "abs": abs, "round": round, "pow": pow}
        result = eval(expression, allowed)  # noqa: S307 — 교육용 제한된 환경
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"


@tool
def get_exchange_rate(currency: str) -> str:
    """현재 환율을 조회합니다. currency는 통화 코드(USD, EUR, JPY 등)입니다."""
    # 모의 환율 데이터
    rates = {"USD": 1350.5, "EUR": 1480.2, "JPY": 9.15, "CNY": 186.3}
    rate = rates.get(currency.upper())
    if rate:
        return json.dumps({"currency": currency.upper(), "rate_krw": rate, "unit": "1 외화 = KRW"})
    return f"'{currency}' 통화 코드를 찾을 수 없습니다. 지원: {list(rates.keys())}"

In [8]:
# 전체 도구 목록
all_tools = [search_knowledge, calculator, get_exchange_rate, add, multiply, divide]
all_tools_by_name = {t.name: t for t in all_tools}

print("사용 가능한 도구:")
for t in all_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

사용 가능한 도구:
  - search_knowledge: SK 하이닉스 내부 지식 베이스를 검색합니다. 반도체, 메모리 제품 정보 등을 조회할 때 사용합니다....
  - calculator: 수학 표현식을 계산합니다. 사칙연산, 거듭제곱(**) 등을 지원합니다....
  - get_exchange_rate: 현재 환율을 조회합니다. currency는 통화 코드(USD, EUR, JPY 등)입니다....
  - add: 두 수를 더합니다....
  - multiply: 두 수를 곱합니다....
  - divide: a를 b로 나눕니다. b가 0이면 에러를 반환합니다....


In [ ]:
# [4-11] : 복합 도구 바인딩 및 단일 호출 테스트
# [핵심] 여러 도구를 한 번에 바인딩 — LLM이 질문에 맞는 도구를 자동 선택
# [주의] 도구 docstring이 모호하면 LLM이 잘못된 도구를 선택할 수 있음

llm_multi_tools = 

# 검색 도구 테스트
resp = 
print("검색 도구 선택:")
for tc in resp.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

검색 도구 선택:
  → search_knowledge({'query': 'HBM3E 메모리'})


In [ ]:
# 환율 도구 테스트
resp2 = 
print("환율 도구 선택:")
for tc in resp2.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

환율 도구 선택:
  → get_exchange_rate({'currency': 'USD'})


---

## 3) ReAct Agent 구현 — tool-calling loop (Graph API)

**ReAct**(Reasoning + Acting) 패턴은 다음 루프를 반복합니다:

1. **LLM 노드** — 현재 메시지를 기반으로 도구 호출 여부를 결정
2. **Tool 노드** — LLM이 선택한 도구를 실제로 실행
3. **조건부 엣지** — `tool_calls`가 있으면 Tool 노드로, 없으면 `END`로 라우팅

```
START → llm → [tool_calls?] → tools → llm → ... → END
```



In [ ]:
# [4-12] : ReAct Agent — StateGraph로 tool-calling 루프 구현
# [핵심] llm → tools → llm 루프가 ReAct 패턴의 핵심
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage, HumanMessage

# 시스템 프롬프트 정의
SYSTEM_PROMPT = (
    "당신은 SK하이닉스의 AI 어시스턴트입니다. "
    "제공된 도구를 사용하여 정확한 정보를 제공하세요. "
    "도구 결과를 바탕으로 자연스러운 한국어로 답변하세요."
)

# 툴 바인딩
tools = [search_knowledge, calculator, get_exchange_rate, add, multiply, divide]
llm_with_tools = 

In [ ]:
# LLM 노드 — 시스템 프롬프트 + 대화 이력을 모델에 전달
def llm_node(state: MessagesState) -> dict:
    
    return 

In [ ]:
# 그래프 조립
builder = 

graph = builder.compile()

In [ ]:
# [4-13] : ReAct Agent 실행 — 단일 도구 호출
# [핵심] graph.invoke()로 전체 루프가 자동 실행됨
# [주의] 결과의 messages[-1]이 최종 AI 답변
result = 

print(result["messages"][-1].content)

최신 내부 문서에 따르면, HBM3E는 최신 고대역폭 메모리(HBM)로 AI 가속기에 필수적인 부품입니다. 이 제품은 최대 36GB의 용량을 지원하며, 고성능 컴퓨팅 환경에서 중요한 역할을 합니다.


In [15]:
# [4-14] : 스트리밍으로 실행 흐름 관찰
# [핵심] stream_mode="updates"로 각 노드 실행을 단계별로 추적
# [주의] 복합 질문은 여러 번의 도구 호출을 발생시킴

print("에이전트 실행 흐름:")
print("=" * 60)

for chunk in graph.stream({"messages": [HumanMessage(content="100달러를 한국 원화로 바꾸면 얼마인가요? 환율을 조회해서 계산해주세요.")]}, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"\n[{node_name}]")
        for msg in update.get("messages", []):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  도구 호출: {tc['name']}({tc['args']})")
            elif hasattr(msg, "content") and msg.content:
                print(f"  {msg.content[:200]}")

에이전트 실행 흐름:

[llm]
  도구 호출: get_exchange_rate({'currency': 'USD'})
  도구 호출: multiply({'a': 100, 'b': 0})

[tools]
  {"currency": "USD", "rate_krw": 1350.5, "unit": "1 \uc678\ud654 = KRW"}
  0.0

[llm]
  도구 호출: multiply({'a': 100, 'b': 1350.5})

[tools]
  135050.0

[llm]
  현재 1달러당 환율이 약 1,350.5원입니다. 따라서 100달러를 한국 원화로 환전하면 약 135,050원이 됩니다. (실제 환전 시에는 환전 수수료가 추가될 수 있습니다.)


In [16]:
# [4-15] : 복합 질문 — 여러 도구를 순차적으로 호출
# [핵심] ReAct 패턴은 도구 결과를 보고 추가 도구 호출을 자동 결정
# [주의] 반복 횟수에 제한이 없으므로 프로덕션에서는 max_iterations 설정 권장

result = graph.invoke({"messages": [HumanMessage(content="SK하이닉스가 어떤 회사인지 검색하고, HBM3E의 최대 용량(GB)에 1024를 곱하면 몇 MB인지 계산해주세요.")]})

# 전체 메시지 흐름 출력
print("전체 대화 흐름:")
print("=" * 60)
for msg in result["messages"]:
    role = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[{role}] 도구 호출: {tc['name']}({tc['args']})")
    elif msg.content:
        print(f"[{role}] {msg.content[:150]}")

print("\n" + "=" * 60)
print("최종 답변:", result["messages"][-1].content)

전체 대화 흐름:
[HumanMessage] SK하이닉스가 어떤 회사인지 검색하고, HBM3E의 최대 용량(GB)에 1024를 곱하면 몇 MB인지 계산해주세요.
[AIMessage] 도구 호출: search_knowledge({'query': 'SK하이닉스 회사 소개'})
[AIMessage] 도구 호출: search_knowledge({'query': 'HBM3E 최대 용량'})
[ToolMessage] 'SK하이닉스 회사 소개'에 대한 검색 결과가 없습니다.
[ToolMessage] HBM3E는 최신 고대역폭 메모리로, AI 가속기에 필수적인 부품입니다. 최대 36GB 용량을 지원합니다.
[AIMessage] 도구 호출: multiply({'a': 36, 'b': 1024})
[ToolMessage] 36864.0
[AIMessage] SK하이닉스는 대한민국을 대표하는 반도체 기업 중 하나로, 주로 DRAM, NAND 플래시 등 메모리 반도체를 개발 및 생산하는 글로벌 기업입니다. 특히 고성능 메모리 분야에서 세계적인 경쟁력을 갖추고 있습니다.

또한, HBM3E는 SK하이닉스가 개발한 최신 고대역폭

최종 답변: SK하이닉스는 대한민국을 대표하는 반도체 기업 중 하나로, 주로 DRAM, NAND 플래시 등 메모리 반도체를 개발 및 생산하는 글로벌 기업입니다. 특히 고성능 메모리 분야에서 세계적인 경쟁력을 갖추고 있습니다.

또한, HBM3E는 SK하이닉스가 개발한 최신 고대역폭 메모리로, 최대 36GB의 용량을 지원합니다. 이 최대 용량(36GB)에 1024를 곱하면 36,864MB가 됩니다.


---

## 4) 에러 핸들링 — checkpointer, state persistence, thread_id

프로덕션 워크플로에서는 **에러가 발생해도 처음부터 다시 시작하지 않는** 것이 핵심입니다.

- **체크포인터(`InMemorySaver`)** — 각 노드 실행 후 상태를 자동 저장
- **Durable Execution** — 실패 시 마지막 성공 체크포인트에서 재개
- **`thread_id`** — 대화/실행 세션을 구분하는 고유 식별자
- **스레드 독립성** — 서로 다른 `thread_id`는 완전히 독립된 상태



In [17]:
# [4-16] : 체크포인터로 멀티턴 대화 에이전트
# [핵심] InMemorySaver를 compile()에 전달 → 대화 상태 자동 저장
# [주의] 같은 thread_id를 사용해야 이전 대화가 유지됨

from langgraph.checkpoint.memory import InMemorySaver

# 체크포인터를 추가하여 다시 컴파일
agent_with_memory = builder.compile(checkpointer=InMemorySaver())

config_session = {"configurable": {"thread_id": "session-hynix-01"}}

In [18]:
# 턴 1
r1 = agent_with_memory.invoke({"messages": [HumanMessage(content="현재 달러 환율을 알려주세요.")]}, config_session)
print("턴 1:", r1["messages"][-1].content)

턴 1: 현재 미국 달러(USD)의 환율은 1달러당 1,350.5원입니다.


In [19]:
# 턴 2 — 이전 대화를 기억
r2 = agent_with_memory.invoke({"messages": [HumanMessage(content="그 환율로 500달러는 얼마인가요?")]}, config_session)
print("턴 2:", r2["messages"][-1].content)

턴 2: 현재 환율로 500달러는 675,250원입니다.


In [20]:
# [4-17] : Durable Execution — 실패 시 재개 패턴
# [핵심] 체크포인터가 있으면 실패한 노드만 다시 실행 — 이미 완료된 노드는 건너뜀
# [주의] 외부 API 호출 등 비결정적 노드에서 특히 유용

from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator


class PipelineState(TypedDict):
    data: str
    log: Annotated[list[str], operator.add]


# 실패 카운터 — 첫 번째 시도에서만 실패
attempt_count = {"step_3": 0}


def step_1(state: PipelineState) -> dict:
    print("  step_1: 데이터 수집 실행")
    return {"data": "수집됨", "log": ["step_1: 완료"]}


def step_2(state: PipelineState) -> dict:
    print("  step_2: 데이터 분석 실행")
    return {"data": "분석됨", "log": ["step_2: 완료"]}


def step_3(state: PipelineState) -> dict:
    attempt_count["step_3"] += 1
    print(f"  step_3: 외부 API 호출 (시도 #{attempt_count['step_3']})")
    if attempt_count["step_3"] < 2:
        raise ConnectionError("외부 API 일시적 장애")
    return {"data": "전송됨", "log": ["step_3: 완료"]}


# 그래프 구성
pipe_builder = StateGraph(PipelineState)
pipe_builder.add_node("step_1", step_1)
pipe_builder.add_node("step_2", step_2)
pipe_builder.add_node("step_3", step_3)
pipe_builder.add_edge(START, "step_1")
pipe_builder.add_edge("step_1", "step_2")
pipe_builder.add_edge("step_2", "step_3")
pipe_builder.add_edge("step_3", END)

pipeline = pipe_builder.compile(checkpointer=InMemorySaver())

config_pipe = {"configurable": {"thread_id": "pipeline-01"}}

# 첫 번째 시도 — step_3에서 실패
print("=== 첫 번째 실행 ===")
try:
    pipeline.invoke({"data": "", "log": []}, config_pipe)
except ConnectionError as e:
    print(f"  오류 발생: {e}")

print()

# 두 번째 시도 — step_1, step_2는 건너뛰고 step_3만 재실행
print("=== 두 번째 실행 (재개) ===")
result = pipeline.invoke(None, config_pipe)  # None → 마지막 체크포인트에서 재개
print(f"\n최종 로그: {result['log']}")

=== 첫 번째 실행 ===
  step_1: 데이터 수집 실행
  step_2: 데이터 분석 실행
  step_3: 외부 API 호출 (시도 #1)
  오류 발생: 외부 API 일시적 장애

=== 두 번째 실행 (재개) ===
  step_3: 외부 API 호출 (시도 #2)

최종 로그: ['step_1: 완료', 'step_2: 완료', 'step_3: 완료']


In [21]:
# [4-18] : 스레드 독립성 확인
# [핵심] 다른 thread_id는 완전히 독립된 상태 — 대화 이력이 공유되지 않음
# [주의] 프로덕션에서 사용자별 세션 분리에 활용

# 새로운 스레드 — 이전 대화를 모름
config_new = {"configurable": {"thread_id": "session-hynix-02"}}

r3 = agent_with_memory.invoke({"messages": [HumanMessage(content="이전에 제가 물어본 환율이 얼마였나요?")]}, config_new)
print("새 스레드 답변:", r3["messages"][-1].content)
print("→ 다른 thread_id이므로 이전 대화(session-hynix-01)를 알 수 없음")

새 스레드 답변: 죄송하지만, 이전에 어떤 환율을 물어보셨는지에 대한 기록이 남아 있지 않습니다. 다시 한 번 원하시는 통화(예: USD, EUR, JPY 등)를 말씀해 주시면 현재 환율을 바로 안내해드릴 수 있습니다. 어떤 환율이 궁금하신가요?
→ 다른 thread_id이므로 이전 대화(session-hynix-01)를 알 수 없음


---

## 5) HITL (Human-in-the-Loop) — Agent 실행 승인 워크플로

프로덕션 Agent에서 **위험한 Tool 실행 전에 사용자 승인**을 받는 것은 필수입니다.

LangGraph는 `interrupt()`와 `Command(resume=...)`로 이를 구현합니다:

| 구성 요소 | 역할 |
|-----------|------|
| **`interrupt(value)`** | 노드 실행 중 그래프를 일시 중단하고, 사용자에게 `value`를 전달 |
| **`Command(resume=value)`** | 중단된 그래프를 `value`와 함께 재개 — `interrupt()`의 반환값이 됨 |
| **`InMemorySaver`** | 중단/재개 사이의 상태를 보존하는 체크포인터 (필수) |

### 적용 시나리오
- 외부 API 호출 전 승인 (결제, 메일 발송 등)
- 위험한 파일 조작 (삭제, 덮어쓰기) 전 확인
- 민감 데이터 접근 시 사용자 인증

In [ ]:
# [4-24] : HITL — 위험한 Tool 실행 전 사용자 승인 노드
# [핵심] interrupt()로 위험한 작업 전에 사용자 승인을 요청하는 패턴
# [주의] checkpointer 없이는 interrupt()가 동작하지 않음
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# 상태 정의
class State(TypedDict):
    


In [ ]:
# HITL를 적용시킬 노드 정의
def review_node(state: State):

    return 

# 사용자 승인 후 노드
def execute_node(state: State):

    return

In [ ]:
builder = 


graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
config = {"configurable": {"thread_id": "thread-1"}}


# 1차 실행: review_node에서 interrupt 발생
result = 
result

{'task': '프로덕션 DB에서 오래된 데이터를 삭제',
 '__interrupt__': [Interrupt(value={'message': '이 작업을 실행해도 될까요?', 'task': '프로덕션 DB에서 오래된 데이터를 삭제', 'options': ['approve', 'reject']}, id='8d2193363f525df1bf97717fb10e0e54')]}

In [1]:
# 사용자 입력 받기
user_input = input("\n실행하시겠습니까? approve / reject 입력: ").strip().lower()
user_input

'approve'

In [ ]:
# 사용자 입력값으로 그래프 재개
resumed = 
resumed

{'task': '프로덕션 DB에서 오래된 데이터를 삭제',
 'approved': True,
 'result': '작업 실행 완료: 프로덕션 DB에서 오래된 데이터를 삭제'}

---

## 6) Agent 디버깅 — state history, interrupt, 스트리밍, 트레이싱

에이전트 워크플로를 디버깅하고 제어하기 위한 핵심 도구:

| 기능 | API | 용도 |
|------|-----|------|
| **상태 이력** | `get_state()`, `get_state_history()` | 체크포인트 확인, 실행 추적 |
| **인터럽트** | `interrupt()`, `Command(resume=...)` | 실행 중단 & 사람의 승인 후 재개 |
| **상태 수정** | `update_state()` | 외부에서 상태를 직접 변경하여 실행 조정 |
| **스트리밍** | `stream(stream_mode="updates")` | 노드별 실행을 실시간으로 관찰 |
| **트레이싱** | LangSmith / Langfuse | 전체 호출 체인을 대시보드에서 시각적 추적 |

In [ ]:
# [4-19] : get_state() & get_state_history() — 실행 상태 추적
# [핵심] 체크포인터가 저장한 모든 상태를 조회하여 디버깅
# [주의] get_state_history()는 최신순 반환 — 인덱스 0이 가장 최근

# 현재 상태 조회
state = agent_with_memory.get_state(config_session)
print("=== 현재 상태 ===")
print(f"스레드: {config_session['configurable']['thread_id']}")
print(f"메시지 수: {len(state.values['messages'])}")
print(f"체크포인트 ID: {state.config['configurable']['checkpoint_id'][:30]}...")
print(f"다음 노드: {state.next}")

=== 현재 상태 ===
스레드: session-hynix-01
메시지 수: 8
체크포인트 ID: 1f16a04d-8de2-632c-8008-1bc609...
다음 노드: ()


In [28]:
# 상태 이력 조회
print("=== 상태 이력 (최신순) ===")
for i, snapshot in enumerate(agent_with_memory.get_state_history(config_session)):
    msg_count = len(snapshot.values.get("messages", []))
    cp_id = snapshot.config["configurable"]["checkpoint_id"][:20]
    print(f"  [{i}] 체크포인트={cp_id}... 메시지={msg_count}")
    if i >= 5:
        print("  ... (이하 생략)")
        break

=== 상태 이력 (최신순) ===
  [0] 체크포인트=1f16a04d-8de2-632c-8... 메시지=8
  [1] 체크포인트=1f16a04d-83f2-6291-8... 메시지=7
  [2] 체크포인트=1f16a04d-83ed-64bb-8... 메시지=6
  [3] 체크포인트=1f16a04d-7c2e-620d-8... 메시지=5
  [4] 체크포인트=1f16a04d-7c2b-66ee-8... 메시지=4
  [5] 체크포인트=1f16a04d-7c0f-68ef-8... 메시지=4
  ... (이하 생략)


In [29]:
# [4-20] : interrupt() — Human-in-the-Loop 패턴
# [핵심] 민감한 작업 전에 사람의 승인을 받고 진행하는 패턴
# [주의] interrupt()는 반드시 checkpointer가 있어야 동작
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# 상태 정의
class State(TypedDict):
    task: str
    approved: bool
    result: str

# HITL를 적용시킬 노드 정의
def review_node(state: State):
    decision = interrupt({
        "message": "이 작업을 실행해도 될까요?",
        "task": state["task"],
        "options": ["approve", "reject"]
    })

    return {"approved": decision == "approve"}

# 사용자 승인 후 노드
def execute_node(state: State):
    if not state["approved"]:
        return {"result": "사용자가 작업을 거절했습니다."}

    return {"result": f"작업 실행 완료: {state['task']}"}

builder = StateGraph(State)
builder.add_node("review", review_node)
builder.add_node("execute", execute_node)

builder.add_edge(START, "review")
builder.add_edge("review", "execute")
builder.add_edge("execute", END)

approval_graph = builder.compile(checkpointer=InMemorySaver())

In [30]:
config = {"configurable": {"thread_id": "thread-1"}}

# 1차 실행: review_node에서 interrupt 발생
result = approval_graph.invoke({"task": "프로덕션 DB에서 오래된 데이터를 삭제"}, config=config)
result

{'task': '프로덕션 DB에서 오래된 데이터를 삭제',
 '__interrupt__': [Interrupt(value={'message': '이 작업을 실행해도 될까요?', 'task': '프로덕션 DB에서 오래된 데이터를 삭제', 'options': ['approve', 'reject']}, id='6b17651420152d8eb41ee9923c8c986b')]}

In [31]:
# 중단 상태 확인
paused_state = approval_graph.get_state(config)
print(f"  다음 노드: {paused_state.next}")
print(f"  인터럽트: {paused_state.tasks[0].interrupts[0].value}")

  다음 노드: ('review',)
  인터럽트: {'message': '이 작업을 실행해도 될까요?', 'task': '프로덕션 DB에서 오래된 데이터를 삭제', 'options': ['approve', 'reject']}


In [32]:
# 사용자 입력 받기
user_input = input("\n실행하시겠습니까? approve / reject 입력: ").strip().lower()

# 사용자 입력값으로 그래프 재개
resumed = graph.invoke(
    Command(resume=user_input),
    config=config
)
resumed

{'task': '프로덕션 DB에서 오래된 데이터를 삭제',
 'approved': True,
 'result': '작업 실행 완료: 프로덕션 DB에서 오래된 데이터를 삭제'}

In [33]:
# [4-22] : update_state() — 외부에서 상태를 직접 수정
# [핵심] 디버깅 또는 수동 개입 시 상태를 프로그래밍 방식으로 변경
# [주의] update_state()는 새 체크포인트를 생성 — 기존 이력은 보존됨


config = {"configurable": {"thread_id": "thread-2"}}

# 1차 실행: review_node에서 interrupt 발생
result = approval_graph.invoke({"task": "프로덕션 DB에서 오래된 데이터를 삭제"}, config=config)
result

{'task': '프로덕션 DB에서 오래된 데이터를 삭제',
 '__interrupt__': [Interrupt(value={'message': '이 작업을 실행해도 될까요?', 'task': '프로덕션 DB에서 오래된 데이터를 삭제', 'options': ['approve', 'reject']}, id='21827506dca60036eb9a58fedebfead5')]}

In [34]:
# 상태에 task 변경
graph.update_state(
    config,
    {"task": "AI Agent 구성 요소는?"},
)

# 사용자 입력 받기
user_input = input("\n실행하시겠습니까? approve / reject 입력: ").strip().lower()

# 사용자 입력값으로 그래프 재개
resumed = graph.invoke(
    Command(resume=user_input),
    config=config
)
resumed

{'task': 'AI Agent 구성 요소는?',
 'approved': True,
 'result': '작업 실행 완료: AI Agent 구성 요소는?'}

In [35]:
# [4-23] : 타임 트래블 — 이전 체크포인트로 되돌아가기
# [핵심] get_state_history()에서 interrupt 지점의 config를 가져와 그 지점에서 다른 결정으로 재개
# [주의] 타임 트래블 후 새 실행은 분기(fork)를 생성 — 기존 이력은 보존

# 동일한 thread_id 안에서 체크포인트 이력이 관리됨
config = {
    "configurable": {
        "thread_id": "hitl-time-travel-demo"
    }
}

# 1) 최초 실행: review_node에서 interrupt 발생
r1 = approval_graph.invoke(
    {
        "task": "민감한 작업 실행",
        "approved": False,
        "result": ""
    },
    config,
)

print("최초 실행 결과:")
print(r1)

if "__interrupt__" in r1:
    interrupt_value = r1["__interrupt__"][0].value
    print("\n인터럽트 발생:")
    print(f"  메시지: {interrupt_value['message']}")
    print(f"  작업: {interrupt_value['task']}")
    print(f"  선택지: {interrupt_value['options']}")

# 2) 일단 approve로 재개해서 정상 완료
r_approve = approval_graph.invoke(
    Command(resume="approve"),
    config,
)

print("\napprove 후 결과:")
print(f"  approved: {r_approve['approved']}")
print(f"  result: {r_approve['result']}")

# 3) 체크포인트 이력 조회
history = list(approval_graph.get_state_history(config))
print(f"\n전체 체크포인트 수: {len(history)}")

# 체크포인트 구조 확인용 출력
for i, h in enumerate(history):
    print(f"\n[{i}] checkpoint_id:", h.config["configurable"]["checkpoint_id"][:30] + "...")
    print("  values:", h.values)
    print("  next:", h.next)

# 4) interrupt가 걸렸던 체크포인트 찾기
target = None

for h in history:
    tasks = getattr(h, "tasks", ()) or ()
    has_interrupt = any(getattr(t, "interrupts", None) for t in tasks)

    if has_interrupt:
        target = h
        break

# fallback: review 노드 재개 지점으로 보이는 체크포인트 찾기
if target is None:
    for h in history:
        if "review" in (getattr(h, "next", ()) or ()):
            target = h
            break

if target is None:
    print("\n되돌아갈 interrupt 체크포인트를 찾지 못했습니다.")
else:
    target_config = target.config

    print("\n되돌아갈 체크포인트:")
    print(f"  checkpoint_id: {target_config['configurable']['checkpoint_id'][:30]}...")
    print(f"  해당 시점 상태: {target.values}")

    # 5) 그 시점에서 이번에는 reject로 재개
    #    기존 approve 이력은 보존되고, reject 경로가 새 분기(fork)로 생성됨
    try:
        r_travel = approval_graph.invoke(
            Command(resume="reject"),
            target_config,
        )

        print("\n타임 트래블 후 reject로 재개한 결과:")
        print(f"  approved: {r_travel['approved']}")
        print(f"  result: {r_travel['result']}")

    except Exception as e:
        print(f"\n타임 트래블 중 오류: {type(e).__name__}: {str(e)[:200]}")

최초 실행 결과:
{'task': '민감한 작업 실행', 'approved': False, 'result': '', '__interrupt__': [Interrupt(value={'message': '이 작업을 실행해도 될까요?', 'task': '민감한 작업 실행', 'options': ['approve', 'reject']}, id='da4b9972b2d51d8a0417d13bbe824895')]}

인터럽트 발생:
  메시지: 이 작업을 실행해도 될까요?
  작업: 민감한 작업 실행
  선택지: ['approve', 'reject']

approve 후 결과:
  approved: True
  result: 작업 실행 완료: 민감한 작업 실행

전체 체크포인트 수: 4

[0] checkpoint_id: 1f16a04d-e04e-686a-8002-0c15bb...
  values: {'task': '민감한 작업 실행', 'approved': True, 'result': '작업 실행 완료: 민감한 작업 실행'}
  next: ()

[1] checkpoint_id: 1f16a04d-e047-6ba4-8001-f2b5f4...
  values: {'task': '민감한 작업 실행', 'approved': True, 'result': ''}
  next: ('execute',)

[2] checkpoint_id: 1f16a04d-e02b-6c3c-8000-c4dda0...
  values: {'task': '민감한 작업 실행', 'approved': False, 'result': ''}
  next: ('review',)

[3] checkpoint_id: 1f16a04d-e02a-623e-bfff-fbbe45...
  values: {}
  next: ('__start__',)

되돌아갈 체크포인트:
  checkpoint_id: 1f16a04d-e02b-6c3c-8000-c4dda0...
  해당 시점 상태: {'task': '민감한 작업 실행', 'app

In [36]:
# [4-28] : 스트리밍 모드를 활용한 실시간 디버깅
# [핵심] stream_mode="updates"로 각 노드 실행을 실시간 관찰

print("=== stream_mode='updates' — 노드별 실행 추적 ===")

debug_config = {"configurable": {"thread_id": "debug-hitl-stream-01"}}

# 1) interrupt() 발생 전까지 실행
for chunk in approval_graph.stream(
    {
        "task": "현재 달러 환율을 검색해주세요",
        "approved": False,
        "result": "",
    },
    debug_config,
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"\n[노드: {node_name}]")
        print(update)


# 2) interrupt() 이후 승인 값 전달해서 재개
for chunk in approval_graph.stream(
    Command(resume="approve"),
    debug_config,
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"\n[노드: {node_name}]")
        print(update)


print("\n=== 스트리밍 디버깅 완료 ===")

=== stream_mode='updates' — 노드별 실행 추적 ===

[노드: __interrupt__]
(Interrupt(value={'message': '이 작업을 실행해도 될까요?', 'task': '현재 달러 환율을 검색해주세요', 'options': ['approve', 'reject']}, id='aae1e1f9622dc8e52ce1173bf49092e8'),)

[노드: review]
{'approved': True}

[노드: execute]
{'result': '작업 실행 완료: 현재 달러 환율을 검색해주세요'}

=== 스트리밍 디버깅 완료 ===


---

## 정리

| 항목 | 내용 |
|------|------|
| **다룬 기술** | 조건 분기, `@tool`, `bind_tools()`, ReAct Agent, 체크포인터, HITL, 스트리밍 디버깅 |
| **핵심 개념** | 상태 기반 동적 라우팅, Tool-calling 루프, Durable Execution, Human-in-the-Loop, Agent 디버깅 |

### 핵심 패턴 요약

| 패턴 | 구현 | 용도 |
|------|------|------|
| **조건 분기** | `add_conditional_edges()` + 라우팅 함수 | 상태에 따른 동적 흐름 제어 |
| **Tool 바인딩** | `@tool` + `bind_tools()` + `ToolMessage` | LLM에 외부 도구 연결 |
| **ReAct 루프** | `llm → [tool_calls?] → tools → llm → END` | 자율적 도구 선택 & 실행 |
| **Durable Execution** | `InMemorySaver` + `thread_id` | 실패 시 재개, 상태 영속성 |
| **HITL** | `interrupt()` + `Command(resume=...)` | 위험한 Tool 실행 전 사용자 승인 |
| **Deep Agents HITL** | `interrupt_on={"tool_name": True}` | 선언적 HITL 설정 |
| **타임 트래블** | `get_state_history()` + 체크포인트 config | 이전 상태로 되돌아가기 |
| **스트리밍 디버깅** | `stream(stream_mode="updates")` | 노드별 실행 실시간 관찰 |
| **트레이싱** | LangSmith / Langfuse | 전체 호출 체인 시각적 추적 |